# Step 1. Tier 1 대안데이터 수집

기존 PJ12 프로젝트의 시세(prices.csv) + 거시지표(macro.csv)에 더해,  
**레짐 감지와 조기 경보**를 강화할 수 있는 Tier 1 대안데이터를 수집합니다.

| 카테고리 | 수집 대상 | 소스 | 의미 |
|----------|----------|------|------|
| VIX 기간구조 | VIX9D, VIX3M, VIX6M | yfinance | 변동성 곡면 — 백워데이션 = 위기 임박 |
| 꼬리 위험 | SKEW Index | yfinance | 블랙스완 우려 수준 |
| 실물 경기 | 구리 선물 (HG=F) | yfinance | Cu/Au 비율 → 경기 확장/수축 |
| 신용 시장 | HY OAS, 10Y-2Y 스프레드 | FRED | 자금 경색, 수익률 곡선 역전 |
| 주간 매크로 | 실업수당, WEI, Sahm Rule | FRED | 월간 지표의 고빈도 대체 |

In [1]:
# ============================================================
# 라이브러리 임포트
# ============================================================
import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader import data as web
import os, warnings

warnings.filterwarnings('ignore')  # yfinance FutureWarning 억제

# ── 경로 설정 ──
BASE_DIR  = os.path.dirname(os.getcwd())          # PJ12_Asset_Simulater/
V2_DIR    = os.path.join(BASE_DIR, 'PJ12_v2_AltData')  # 현재 폴더
DATA_DIR  = os.path.join(V2_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# ── 분석 기간 (기존 프로젝트와 동일) ──
START = '2021-01-01'
END   = '2025-12-31'

print(f'분석 기간: {START} ~ {END}')
print(f'저장 경로: {DATA_DIR}')

분석 기간: 2021-01-01 ~ 2025-12-31
저장 경로: C:\Users\gorhk\DA_Portfolio\01_Daily_Project\PJ12_Asset_Simulater\PJ12_v2_AltData\data


## 1-1. 기존 데이터 로드

원본 프로젝트의 `prices.csv`와 `macro.csv`를 로드하여 날짜 범위를 확인합니다.

In [2]:
# ============================================================
# 기존 prices.csv, macro.csv 로드 (원본 프로젝트에서 생성된 파일)
# ============================================================
df_prices = pd.read_csv(
    os.path.join(BASE_DIR, 'prices.csv'),
    index_col=0, parse_dates=True
)
df_macro = pd.read_csv(
    os.path.join(BASE_DIR, 'macro.csv'),
    index_col=0, parse_dates=True
)

print(f'prices.csv : {df_prices.shape}  |  기간: {df_prices.index[0].date()} ~ {df_prices.index[-1].date()}')
print(f'macro.csv  : {df_macro.shape}  |  기간: {df_macro.index[0].date()} ~ {df_macro.index[-1].date()}')
print(f'\nprices 컬럼: {list(df_prices.columns)}')
print(f'macro 컬럼 : {list(df_macro.columns)}')

prices.csv : (1825, 13)  |  기간: 2021-01-01 ~ 2025-12-30
macro.csv  : (1320, 3)  |  기간: 2021-01-01 ~ 2025-12-31

prices 컬럼: ['SPY', 'QQQ', 'TLT', 'AGG', 'GLD', 'EEM', 'CL=F', 'GC=F', 'SI=F', 'BTC-USD', 'ETH-USD', '^VIX', 'DX-Y.NYB']
macro 컬럼 : ['DGS10', 'CPIAUCSL', 'UNRATE']


## 1-2. yfinance — VIX 기간구조 + SKEW + 구리 선물 수집

| 티커 | 설명 | 활용 |
|------|------|------|
| ^VIX9D | VIX 9일 (초단기 내재 변동성) | 기간구조 단기 끝 |
| ^VIX3M | VIX 3개월 | 기간구조 중기 — VIX3M/VIX 비율로 콘탱고/백워데이션 판단 |
| ^VIX6M | VIX 6개월 | 기간구조 장기 끝 |
| ^SKEW | CBOE SKEW Index | 꼬리 위험 프리미엄 (높을수록 좌측 꼬리 우려 큼) |
| HG=F | 구리 선물 | Cu/Au 비율 → 경기 확장(비율 상승)/수축(비율 하락) |

In [3]:
# ============================================================
# yfinance 신규 티커 수집
# ============================================================

# 수집할 대안데이터 티커 정의
ALT_YFINANCE = {
    '^VIX9D':  'VIX 9-Day (초단기 내재 변동성)',
    '^VIX3M':  'VIX 3-Month (중기 내재 변동성)',
    '^VIX6M':  'VIX 6-Month (장기 내재 변동성)',
    '^SKEW':   'CBOE SKEW Index (꼬리 위험)',
    'HG=F':    'Copper Futures (구리 선물)',
}

# 티커별 개별 수집 (일부 티커 실패 시에도 나머지 수집 가능)
yf_data = {}

for ticker, desc in ALT_YFINANCE.items():
    try:
        df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
        if df.empty:
            print(f'  [WARN] {ticker} ({desc}) — 데이터 없음, 건너뜀')
            continue
        # Close 가격 추출 (MultiIndex 처리)
        if isinstance(df.columns, pd.MultiIndex):
            series = df[('Close', ticker)].dropna()
        else:
            series = df['Close'].dropna()
        yf_data[ticker] = series
        print(f'  [OK] {ticker:10s} ({desc}) — {len(series):,}일 수집')
    except Exception as e:
        print(f'  [ERR] {ticker} ({desc}) — {e}')

# DataFrame으로 병합
df_yf_alt = pd.DataFrame(yf_data)
df_yf_alt.index = pd.to_datetime(df_yf_alt.index)
df_yf_alt = df_yf_alt.sort_index()

print(f'\nyfinance 수집 완료: {df_yf_alt.shape}')
df_yf_alt.tail(3)

  [OK] ^VIX9D     (VIX 9-Day (초단기 내재 변동성)) — 1,254일 수집


  [OK] ^VIX3M     (VIX 3-Month (중기 내재 변동성)) — 1,254일 수집


  [OK] ^VIX6M     (VIX 6-Month (장기 내재 변동성)) — 1,254일 수집


  [OK] ^SKEW      (CBOE SKEW Index (꼬리 위험)) — 1,214일 수집


  [OK] HG=F       (Copper Futures (구리 선물)) — 1,257일 수집

yfinance 수집 완료: (1257, 5)


,^VIX9D,^VIX3M,^VIX6M,^SKEW,HG=F
Date,,,,,
2025-12-26,9.65,17.77,20.639999,151.449997,5.7665
2025-12-29,11.37,17.82,20.600000,150.470001,5.4905
2025-12-30,11.46,17.77,20.690001,148.330002,5.7275


## 1-3. FRED — 신용 스프레드 + 주간 매크로 Nowcasting 수집

| 시리즈 ID | 설명 | 빈도 | 활용 |
|-----------|------|------|------|
| BAMLH0A0HYM2 | ICE BofA HY OAS (하이일드 신용스프레드) | 일간 | 신용 시장 스트레스 |
| T10Y2Y | 10Y-2Y Treasury Spread | 일간 | 수익률 곡선 역전 = 경기침체 신호 |
| ICSA | 신규 실업수당 청구 | 주간 | 고용 악화의 가장 빠른 신호 |
| WEI | Weekly Economic Index (주간 경제 활동) | 주간 | 실시간 경기 추적 |
| SAHMREALTIME | Sahm 경기침체 지표 | 월간 | 실업률 3개월 MA 급등 → 침체 확정 |

In [4]:
# ============================================================
# FRED 신규 시리즈 수집
# ============================================================

ALT_FRED = {
    'BAMLH0A0HYM2': 'HY OAS (하이일드 신용스프레드, 일간)',
    'T10Y2Y':        '10Y-2Y 수익률 곡선 스프레드 (일간)',
    'ICSA':          '신규 실업수당 청구 (주간)',
    'WEI':           'Weekly Economic Index (주간)',
    'SAHMREALTIME':  'Sahm 경기침체 지표 (월간)',
}

fred_data = {}

for series_id, desc in ALT_FRED.items():
    try:
        s = web.DataReader(series_id, 'fred', START, END)[series_id]
        fred_data[series_id] = s.dropna()
        print(f'  [OK] {series_id:20s} ({desc}) — {len(s.dropna()):,}개 관측치')
    except Exception as e:
        print(f'  [ERR] {series_id:20s} ({desc}) — {e}')

# DataFrame으로 병합
df_fred_alt = pd.DataFrame(fred_data)
df_fred_alt.index = pd.to_datetime(df_fred_alt.index)
df_fred_alt = df_fred_alt.sort_index()

print(f'\nFRED 수집 완료: {df_fred_alt.shape}')
df_fred_alt.tail(3)

  [OK] BAMLH0A0HYM2         (HY OAS (하이일드 신용스프레드, 일간)) — 1,307개 관측치
  [OK] T10Y2Y               (10Y-2Y 수익률 곡선 스프레드 (일간)) — 1,249개 관측치


  [OK] ICSA                 (신규 실업수당 청구 (주간)) — 261개 관측치


  [OK] WEI                  (Weekly Economic Index (주간)) — 261개 관측치


  [OK] SAHMREALTIME         (Sahm 경기침체 지표 (월간)) — 59개 관측치

FRED 수집 완료: (1573, 5)


,BAMLH0A0HYM2,T10Y2Y,ICSA,WEI,SAHMREALTIME
DATE,,,,,
2025-12-29,2.87,0.67,NaN,NaN,NaN
2025-12-30,2.84,0.69,NaN,NaN,NaN
2025-12-31,2.81,0.71,NaN,NaN,NaN


## 1-4. NYSE 영업일 정렬 + 결측치 처리

모든 데이터를 NYSE 영업일 캘린더에 맞추고, 전방/후방 채움(ffill → bfill)으로 결측치를 처리합니다.  
주간/월간 FRED 데이터는 발표 이전 날짜에 이전 값이 유지되므로, **look-ahead bias 없이** ffill이 적절합니다.

In [5]:
# ============================================================
# NYSE 영업일 기준 정렬 + ffill/bfill
# ============================================================

# 기존 프로젝트와 동일한 NYSE 영업일 캘린더
nyse_dates = pd.bdate_range(start=START, end=END, freq='B')
print(f'NYSE 영업일 수: {len(nyse_dates):,}일')

# ── yfinance 데이터 정렬 ──
df_yf_aligned = df_yf_alt.reindex(nyse_dates)
df_yf_aligned = df_yf_aligned.ffill().bfill()

# ── FRED 데이터 정렬 ──
# 주간/월간 FRED 시리즈(ICSA, WEI 등)는 토요일 등 비영업일에 발표됨
# → 먼저 달력일(calendar day) 기준 ffill → 그 다음 NYSE 영업일 필터
# 이렇게 해야 주간 데이터가 영업일에 올바르게 전파됨
daily_range = pd.date_range(start=START, end=END, freq='D')  # 달력일 전체
df_fred_daily = df_fred_alt.reindex(daily_range).ffill().bfill()  # 달력일 기준 채움
df_fred_aligned = df_fred_daily.reindex(nyse_dates)  # NYSE 영업일만 필터

# ── 통합 DataFrame 생성 ──
df_alt_tier1 = pd.concat([df_yf_aligned, df_fred_aligned], axis=1)
df_alt_tier1.index.name = 'Date'

print(f'\n통합 DataFrame: {df_alt_tier1.shape}')
print(f'컬럼: {list(df_alt_tier1.columns)}')

NYSE 영업일 수: 1,304일

통합 DataFrame: (1304, 10)
컬럼: ['^VIX9D', '^VIX3M', '^VIX6M', '^SKEW', 'HG=F', 'BAMLH0A0HYM2', 'T10Y2Y', 'ICSA', 'WEI', 'SAHMREALTIME']


In [6]:
# ============================================================
# 결측률 검증 리포트
# ============================================================

missing_report = pd.DataFrame({
    '컬럼': df_alt_tier1.columns,
    '총 행수': len(df_alt_tier1),
    '결측 수': df_alt_tier1.isnull().sum().values,
    '결측률(%)': (df_alt_tier1.isnull().sum().values / len(df_alt_tier1) * 100).round(2),
})

print('=== 결측률 리포트 ===')
print(missing_report.to_string(index=False))

# 목표: 모든 컬럼 결측률 0% (ffill/bfill 적용 후)
total_missing = df_alt_tier1.isnull().sum().sum()
if total_missing == 0:
    print(f'\n결측치 0개 — 모든 컬럼 정상 채움 완료')
else:
    print(f'\n[주의] 잔여 결측치 {total_missing}개 존재')

=== 결측률 리포트 ===
          컬럼  총 행수  결측 수  결측률(%)
      ^VIX9D  1304     0     0.0
      ^VIX3M  1304     0     0.0
      ^VIX6M  1304     0     0.0
       ^SKEW  1304     0     0.0
        HG=F  1304     0     0.0
BAMLH0A0HYM2  1304     0     0.0
      T10Y2Y  1304     0     0.0
        ICSA  1304     0     0.0
         WEI  1304     0     0.0
SAHMREALTIME  1304     0     0.0

결측치 0개 — 모든 컬럼 정상 채움 완료


In [7]:
# ============================================================
# 기초 통계량 확인
# ============================================================

desc = df_alt_tier1.describe().T
desc['range'] = desc['max'] - desc['min']
print('=== Tier 1 대안데이터 기초 통계량 ===')
desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'range']].round(2)

=== Tier 1 대안데이터 기초 통계량 ===


,count,mean,std,min,25%,50%,75%,max,range
^VIX9D,1304.0,18.22,6.49,9.17,13.57,16.51,21.18,67.63,58.46
^VIX3M,1304.0,21.53,4.71,13.95,18.09,20.78,24.44,41.50,27.55
^VIX6M,1304.0,23.25,4.33,16.20,19.81,22.85,26.23,35.77,19.57
^SKEW,1304.0,140.55,13.31,110.34,132.74,140.94,149.33,183.12,72.78
HG=F,1304.0,4.23,0.48,3.21,3.83,4.23,4.53,5.80,2.58
BAMLH0A0HYM2,1304.0,3.62,0.70,2.59,3.11,3.38,4.10,5.99,3.40
T10Y2Y,1304.0,0.17,0.67,-1.08,-0.40,0.15,0.58,1.59,2.67
ICSA,1304.0,270189.42,132408.92,190000.00,214000.00,224000.00,240000.00,890000.00,700000.00
WEI,1304.0,3.09,2.21,-0.91,1.83,2.26,3.46,10.57,11.48
SAHMREALTIME,1304.0,0.25,0.61,-0.37,-0.03,0.13,0.35,3.03,3.40


## 1-5. CSV 저장

수집한 Tier 1 대안데이터를 `data/alt_tier1.csv`로 저장합니다.  
이후 Step2(Feature Engineering)에서 이 파일을 읽어 파생 변수를 생성합니다.

In [8]:
# ============================================================
# CSV 저장
# ============================================================

save_path = os.path.join(DATA_DIR, 'alt_tier1.csv')
df_alt_tier1.to_csv(save_path)

# 저장 확인
file_size_kb = os.path.getsize(save_path) / 1024
print(f'저장 완료: {save_path}')
print(f'파일 크기: {file_size_kb:.0f} KB')
print(f'Shape: {df_alt_tier1.shape}  ({df_alt_tier1.shape[0]}일 x {df_alt_tier1.shape[1]}개 지표)')
print(f'기간: {df_alt_tier1.index[0].date()} ~ {df_alt_tier1.index[-1].date()}')

저장 완료: C:\Users\gorhk\DA_Portfolio\01_Daily_Project\PJ12_Asset_Simulater\PJ12_v2_AltData\data\alt_tier1.csv
파일 크기: 168 KB
Shape: (1304, 10)  (1304일 x 10개 지표)
기간: 2021-01-01 ~ 2025-12-31
